# Conecta: vender más, ¿conservar el margen?
**Analítica de datos · Universidad Católica de Pereira**  
Andrés Felipe Torres Cano

La administradora de Conecta necesita comparar las ventas y el margen de Tienda y Web a lo largo del tiempo. Un buen mes de ingresos no implica necesariamente una mejora de rentabilidad: los descuentos y la mezcla de productos también importan.

Investigaremos dos años completos de datos simulados, desde noviembre de 2024 hasta octubre de 2026. Cada fila representa una venta de un producto, con una o varias unidades. Los importes están en COP, sin IVA. No hay devoluciones ni gastos operativos; la utilidad calculada es bruta.

Construiremos gráficos con Plotly y un dashboard en Jupyter con controles de canal y rango de meses. Los indicadores y gráficos se recalcularán juntos cada vez que cambie la selección.

Ejecuta las celdas en orden con **Shift + Enter**. Necesitas pandas, numpy, Plotly e ipywidgets. Si falta alguna biblioteca, ejecuta `%pip install pandas numpy plotly ipywidgets` y reinicia la sesión. Los controles requieren mantener activo el kernel de Jupyter. Los datos se generan dentro del cuaderno con una semilla fija.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
import ipywidgets as widgets

# Visor interactivo para JupyterLab.
pio.renderers.default = "plotly_mimetype"
px.defaults.template = "plotly_white"
pd.options.display.float_format = "{:,.2f}".format
print("Plotly", plotly.__version__)

## Una pregunta recurrente necesita una vista consistente
La audiencia es la administradora comercial. Su pregunta mensual es: **¿cómo evolucionan las ventas y el margen, y qué productos concentran los ingresos de cada canal?**

Un dashboard de seguimiento usa definiciones constantes para revisar el mismo negocio a lo largo del tiempo. También puede ayudar a explorar: un filtro permite pasar del total a un canal y formular preguntas más concretas.

La vista tendrá cuatro indicadores —ventas netas, utilidad bruta, margen y número de ventas registradas— y tres gráficos: evolución de ventas, evolución del margen y productos principales. El período y el canal activo permanecerán visibles. Seleccionaremos ambos antes de calcular los indicadores.

## Reunir los registros
Simulamos una expansión gradual de Web, descuentos que cambian con el tiempo y una mayor actividad al final de cada año. Estos supuestos permiten practicar comparaciones mensuales y entre años; no demuestran efectos de una campaña real.

En una empresa, esta celda se reemplazaría por una lectura del sistema de ventas. El resto del análisis partiría de la misma estructura.

In [ ]:
# Simulación reproducible: no son registros de una empresa real.
rng = np.random.default_rng(42)
catalogo = pd.DataFrame([
    ("Portátil", "Equipos", 2400000, 1980000),
    ("Teléfono", "Equipos", 1500000, 1125000),
    ("Tableta", "Equipos", 1100000, 770000),
    ("Monitor", "Equipos", 850000, 552500),
    ("Teclado", "Accesorios", 130000, 65000),
    ("Mouse", "Accesorios", 80000, 32000),
    ("Audífonos", "Accesorios", 180000, 81000),
    ("Webcam", "Accesorios", 200000, 100000),
], columns=["producto", "categoria", "precio_unitario", "costo_unitario"])
registros = []
for fecha in pd.date_range("2024-11-01", "2026-10-31"):
    mes = (fecha.year - 2024) * 12 + fecha.month - 11
    for canal in ["Tienda", "Web"]:
        # En Web crecen las operaciones y aumentan las rebajas durante el período.
        estacionalidad = 1.3 if fecha.month in [11, 12] else 1.0
        cantidad = int(rng.poisson((4 + mes * (0.22 if canal == "Web" else 0.07)) * estacionalidad))
        for _ in range(cantidad):
            producto = catalogo.iloc[int(rng.integers(len(catalogo)))]
            descuento = int(rng.choice([0, 5, 10]))
            if canal == "Web":
                descuento += min(mes // 4, 5)
            registros.append({
                "fecha": fecha, "canal": canal,
                **producto.to_dict(), "unidades": int(rng.integers(1, 4)),
                "descuento_pct": descuento
            })
ventas = pd.DataFrame(registros)
ventas.insert(0, "venta_id", [f"V{i:05d}" for i in range(1, len(ventas) + 1)])
display(ventas.head())
print("Registros:", len(ventas), "| Productos:", ventas["producto"].nunique())

| Campo | Significado |
| --- | --- |
| `venta_id`, `fecha` | Venta registrada y fecha |
| `canal` | Tienda o Web |
| `producto`, `categoria` | Artículo y grupo comercial |
| `unidades` | Cantidad de unidades de esa venta |
| `precio_unitario`, `costo_unitario` | Precio antes de descuento y costo de cada unidad, en COP |
| `descuento_pct` | Porcentaje de rebaja: 5 significa 5 % |

Antes de dibujar, comprobamos registros únicos, faltantes y rangos. Un indicador correcto sobre una tabla duplicada también puede producir una conclusión equivocada.

In [ ]:
assert ventas["venta_id"].is_unique
assert not ventas.isna().any().any()
assert ventas["unidades"].gt(0).all()
assert ventas["descuento_pct"].between(0, 100, inclusive="left").all()
assert ventas[["precio_unitario", "costo_unitario"]].ge(0).all().all()
print("Período observado:", ventas.fecha.min().date(), "a", ventas.fecha.max().date())
display(ventas.groupby("canal").agg(ventas_registradas=("venta_id", "count"), unidades=("unidades", "sum")))

## Acordar qué significa cada indicador
La venta neta de cada fila es `unidades × precio × (1 − descuento / 100)`. El costo de ventas es `unidades × costo unitario`. Su diferencia es la utilidad bruta.

El **margen bruto** del conjunto se calcula como `utilidad bruta total / ventas netas totales × 100`. No promediamos los porcentajes de cada fila: ventas pequeñas y grandes no deben recibir el mismo peso.

El número de ventas registradas cuenta filas porque en este caso cada identificador representa una venta única de un producto. No mide clientes únicos. Un mes sin ventas tiene margen indefinido, no margen cero.

In [ ]:
ventas["venta_neta"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento_pct"] / 100)
ventas["cogs"] = ventas["unidades"] * ventas["costo_unitario"]
ventas["utilidad_bruta"] = ventas["venta_neta"] - ventas["cogs"]
ventas["mes"] = ventas["fecha"].dt.to_period("M").dt.to_timestamp()
MESES = pd.date_range(ventas["mes"].min(), ventas["mes"].max(), freq="MS")
CANALES = ["Todos", "Tienda", "Web"]
PERIODO = "Noviembre 2024–octubre 2026"

## Una línea para observar el cambio
Agrupamos por mes y canal. Plotly Express construye la primera vista con pocas líneas. Al pasar el cursor sobre un punto aparecen sus datos; la leyenda permite ocultar una serie y el doble clic sobre el área restablece el zoom.

In [ ]:
mensual_canal = ventas.groupby(["mes", "canal"], as_index=False)[["venta_neta", "utilidad_bruta"]].sum()
mensual_canal["margen_pct"] = 100 * mensual_canal["utilidad_bruta"] / mensual_canal["venta_neta"]
fig_ventas = px.line(mensual_canal, x="mes", y="venta_neta", color="canal", markers=True,
    labels={"mes": "Mes", "venta_neta": "Ventas netas (COP)", "canal": "Canal"},
    title="¿Qué canal aporta al crecimiento?")
fig_ventas.update_yaxes(rangemode="tozero")
fig_ventas.show()

## El ingreso necesita una segunda lectura
Un crecimiento de ventas no basta para saber qué ocurrió con la rentabilidad bruta. Observamos el margen por canal sobre el mismo período. Separamos dinero y porcentaje en gráficos distintos para que sus unidades sean claras.

In [ ]:
fig_margen = px.line(mensual_canal, x="mes", y="margen_pct", color="canal", markers=True,
    labels={"mes": "Mes", "margen_pct": "Margen bruto (%)", "canal": "Canal"},
    title="¿El margen acompaña el crecimiento de ventas?")
fig_margen.update_yaxes(range=[0, 60], ticksuffix="%")
fig_margen.show()
display(mensual_canal.round({"venta_neta": 2, "utilidad_bruta": 2, "margen_pct": 2}))

## Aplicar canal y período a los mismos registros
Un filtro de fecha no debe limitarse a hacer zoom en el gráfico. También debe cambiar las ventas, la utilidad, el margen y el conteo de las tarjetas.

La función siguiente aplica ambas condiciones antes de resumir. El mes final está incluido por completo: elegimos desde el primer día del mes inicial hasta antes del primer día del mes siguiente al final.

In [ ]:
def filtrar(datos, canal="Todos", inicio=None, fin=None):
    seleccion = datos.copy()
    if canal != "Todos":
        seleccion = seleccion.loc[seleccion["canal"].eq(canal)]
    if inicio is not None:
        seleccion = seleccion.loc[seleccion["fecha"] >= pd.Timestamp(inicio)]
    if fin is not None:
        limite = pd.Timestamp(fin) + pd.offsets.MonthBegin(1)
        seleccion = seleccion.loc[seleccion["fecha"] < limite]
    return seleccion.copy()

In [ ]:
web = filtrar(ventas, "Web")
print("Ventas registradas en Web:", len(web))
display(web.head(3))

## Resumir la selección
Reunimos los cálculos en una función. Devuelve indicadores, una serie mensual y los cinco productos principales de la selección.

El rango mensual se puede pasar a la función para conservar todos los meses elegidos. Un mes sin ventas tiene suma cero, pero margen indefinido. El margen del período se calcula sobre sus totales, no promediando los márgenes mensuales.

In [ ]:
def resumir(datos, meses=None):
    if meses is None:
        meses = MESES
    ingresos = datos["venta_neta"].sum()
    utilidad = datos["utilidad_bruta"].sum()
    margen = 100 * utilidad / ingresos if ingresos > 0 else None
    mensual = datos.groupby("mes")[["venta_neta", "utilidad_bruta"]].sum().reindex(meses, fill_value=0)
    # Un mes sin ventas tiene margen indefinido, no margen cero.
    mensual["margen_pct"] = mensual["utilidad_bruta"].div(mensual["venta_neta"].replace(0, np.nan)).mul(100)
    productos = datos.groupby("producto")["venta_neta"].sum().sort_values(ascending=False).head(5)
    return {"ingresos": ingresos, "utilidad": utilidad, "margen": margen,
            "registros": len(datos), "mensual": mensual, "productos": productos}

In [ ]:
resumen_web = resumir(web)
display(pd.Series({k: resumen_web[k] for k in ["ingresos", "utilidad", "margen", "registros"]}))
display(resumen_web["productos"].to_frame("Ventas netas (COP)"))

## De varias figuras a un dashboard
Usaremos `graph_objects` para combinar cuatro tarjetas y tres gráficos. `make_subplots` organiza la lectura: indicadores arriba, evolución en el centro y productos abajo.

La función recibe registros ya filtrados. El control de Jupyter llamará a esta función cuando cambiemos el canal o las fechas. Así todos los elementos muestran el mismo conjunto de ventas.

Las tarjetas monetarias expresan millones de COP. El cursor sobre las series y barras muestra los valores completos. Las escalas de ventas se mantienen entre selecciones para facilitar la comparación.

In [ ]:
def construir_dashboard(datos, etiqueta="Todo el catálogo", meses=None):
    if meses is None:
        meses = MESES
    periodo = f"{meses[0]:%Y-%m} a {meses[-1]:%Y-%m}"
    if datos.empty:
        fig = go.Figure()
        fig.add_annotation(text="Sin datos para esta selección", showarrow=False)
        fig.update_layout(template="plotly_white", title=etiqueta,
                          xaxis_visible=False, yaxis_visible=False)
        return fig

    fig = make_subplots(
        rows=3, cols=4,
        specs=[[{"type": "indicator"}] * 4,
               [{"type": "xy", "colspan": 2}, None, {"type": "xy", "colspan": 2}, None],
               [{"type": "xy", "colspan": 4}, None, None, None]],
        row_heights=[0.18, 0.38, 0.44], vertical_spacing=0.13,
        horizontal_spacing=0.10,
        subplot_titles=["", "", "", "", "Ventas por mes", "Margen bruto por mes", "Cinco productos con mayores ventas"]
    )
    r = resumir(datos, meses)
    # Millones en las tarjetas; COP completos disponibles en tablas y hover.
    indicadores = [
        ("Ventas netas · millones COP", r["ingresos"] / 1e6, ".1f", ""),
        ("Utilidad bruta · millones COP", r["utilidad"] / 1e6, ".1f", ""),
        ("Margen bruto", r["margen"], ".1f", "%"),
        ("Ventas registradas", r["registros"], ",.0f", "")
    ]
    for col, (titulo, valor, formato, sufijo) in enumerate(indicadores, 1):
        fig.add_trace(go.Indicator(
            mode="number", value=valor,
            title={"text": titulo, "font": {"size": 13}},
            number={"valueformat": formato, "suffix": sufijo, "font": {"size": 31}}), row=1, col=col)
    m = r["mensual"]
    fig.add_trace(go.Scatter(
        x=m.index, y=m["venta_neta"], mode="lines+markers",
        line={"color": "#007F5F", "width": 3}, name="Ventas netas",
        hovertemplate="%{x|%Y-%m}<br>Ventas: %{y:,.0f} COP<extra></extra>"), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=m.index, y=m["margen_pct"], mode="lines+markers",
        line={"color": "#8052A3", "width": 3}, name="Margen bruto",
        hovertemplate="%{x|%Y-%m}<br>Margen: %{y:.2f}%<extra></extra>"), row=2, col=3)
    productos = r["productos"].sort_values()
    fig.add_trace(go.Bar(
        x=productos, y=productos.index, orientation="h",
        marker_color="#1677A5", name="Ventas por producto",
        hovertemplate="%{y}<br>Ventas: %{x:,.0f} COP<extra></extra>"), row=3, col=1)

    max_ventas = ventas.groupby("mes")["venta_neta"].sum().max()
    max_producto = ventas.groupby("producto")["venta_neta"].sum().max()
    fig.update_layout(
        title={"text": f"Conecta | {etiqueta}<br><sup>{periodo} · Datos simulados · Valores sin IVA</sup>", "x": 0.04, "font": {"size": 21}},
        template="plotly_white", height=850, showlegend=False,
        margin={"t": 120, "b": 55, "l": 115, "r": 45},
        font={"family": "Arial", "size": 13},

    )
    # Mismas escalas al cambiar canal: la comparación visual conserva su referencia.
    fig.update_yaxes(title_text="Ventas (COP)", range=[0, max_ventas * 1.15], row=2, col=1)
    fig.update_yaxes(title_text="Margen (%)", range=[0, 60], row=2, col=3)
    fig.update_xaxes(tickformat="%Y-%m", tickangle=-25, row=2, col=1)
    fig.update_xaxes(tickformat="%Y-%m", tickangle=-25, row=2, col=3)
    fig.update_xaxes(title_text="Ventas netas (COP)", range=[0, max_producto * 1.12], row=3, col=1)
    fig.update_yaxes(categoryorder="array", categoryarray=productos.index, row=3, col=1)
    return fig

## Consultar el negocio
Elige **Todos**, **Tienda** o **Web** y arrastra los extremos del rango para seleccionar los meses inicial y final. Ambos meses se incluyen completos. Al soltar el control se recalculan las cuatro tarjetas, las series y el ranking de productos.

Primero compara Tienda y Web sobre un mismo período. Después mantén un canal y cambia el rango: observa que las tarjetas también cambian. Al comparar totales de períodos, considera su duración; doce meses suelen acumular más ventas que tres.

Usamos `responsive=False`, la configuración que mantiene estable la visualización en esta sesión de Jupyter.

In [ ]:
config = {"displaylogo": False, "responsive": False}
control_canal = widgets.Dropdown(options=CANALES, value="Todos", description="Canal:")
control_fechas = widgets.SelectionRangeSlider(
    options=[(mes.strftime("%Y-%m"), mes) for mes in MESES],
    index=(0, len(MESES) - 1), description="Meses:",
    continuous_update=False, layout=widgets.Layout(width="95%"),
    style={"description_width": "70px"}
)

def actualizar_vista(canal, rango):
    inicio, fin = rango
    seleccion = filtrar(ventas, canal, inicio, fin)
    meses = pd.date_range(inicio, fin, freq="MS")
    print(f"Canal: {canal} | Desde {inicio:%Y-%m} hasta {fin:%Y-%m}, ambos incluidos")
    if seleccion.empty:
        print("No hay ventas en esta selección.")
        return
    figura = construir_dashboard(seleccion, f"Canal: {canal}", meses)
    figura.show(renderer="plotly_mimetype", config=config)

salida_dashboard = widgets.interactive_output(
    actualizar_vista, {"canal": control_canal, "rango": control_fechas}
)
salida_dashboard.layout.min_height = "900px"
display(widgets.VBox([control_canal, control_fechas, salida_dashboard]))

## Comparar el dashboard con los datos
La tabla siguiente toma el rango elegido en el control y calcula los indicadores de cada canal. Ejecútala después de cambiar el período y compara la fila del canal activo con sus tarjetas.

La tabla conserva COP completos; las tarjetas redondean millones de COP. El margen de Todos se calcula sobre los totales, no como promedio simple de los márgenes de Tienda y Web.

In [ ]:
inicio, fin = control_fechas.value
ventas_periodo = filtrar(ventas, "Todos", inicio, fin)
print(f"Período de la tabla: {inicio:%Y-%m} a {fin:%Y-%m}")
comparacion_canales = ventas_periodo.groupby("canal").agg(
    ventas_netas=("venta_neta", "sum"),
    utilidad_bruta=("utilidad_bruta", "sum"),
    ventas_registradas=("venta_id", "count")
)
comparacion_canales.loc["Todos"] = comparacion_canales.sum()
comparacion_canales["margen_pct"] = (
    100 * comparacion_canales["utilidad_bruta"]
    / comparacion_canales["ventas_netas"]
)
comparacion_canales["ventas_registradas"] = comparacion_canales["ventas_registradas"].astype(int)
display(comparacion_canales.rename(columns={
    "ventas_netas": "Ventas netas (COP)",
    "utilidad_bruta": "Utilidad bruta (COP)",
    "ventas_registradas": "Ventas registradas",
    "margen_pct": "Margen bruto (%)"
}).round(2))


## Regresar a la decisión
Cuantificamos el cambio entre el primer mes y el último del rango seleccionado para cada canal. Ejecuta esta celda después de mover el rango. La comparación de extremos complementa las series completas; no describe por sí sola todos los cambios intermedios.

In [ ]:
inicio, fin = control_fechas.value
meses = pd.date_range(inicio, fin, freq="MS")
comparacion = []
for canal in ["Tienda", "Web"]:
    m = resumir(filtrar(ventas, canal, inicio, fin), meses)["mensual"]
    inicial, final = m.iloc[0], m.iloc[-1]
    crecimiento = 100 * (final["venta_neta"] / inicial["venta_neta"] - 1) if inicial["venta_neta"] > 0 else np.nan
    comparacion.append({"canal": canal,
        "crecimiento_ventas_pct": crecimiento,
        "margen_inicial_pct": inicial["margen_pct"],
        "margen_final_pct": final["margen_pct"],
        "cambio_margen_puntos": final["margen_pct"] - inicial["margen_pct"]})
print(f"Comparación: {inicio:%Y-%m} frente a {fin:%Y-%m}")
if inicio == fin:
    print("Se seleccionó un solo mes: los extremos son iguales.")
display(pd.DataFrame(comparacion).round(2))

El rango elegido modifica la conclusión: un cambio observado entre dos meses puede convivir con variaciones intermedias o estacionales. El selector permite contrastar períodos comparables antes de recomendar una acción.

Una caída de margen justifica revisar descuentos y mezcla de productos, pero no demuestra que los descuentos hayan causado el crecimiento. También necesitaríamos gastos por canal, devoluciones y disponibilidad. La utilidad bruta todavía no es utilidad neta.

La diferencia entre dos márgenes se expresa en puntos porcentuales. Para comparar ingresos de períodos con distinta duración, usa meses equivalentes o medidas por mes.

## Fuentes para continuar
- [Plotly Express](https://plotly.com/python/plotly-express/)
- [Controles interactivos de Jupyter](https://ipywidgets.readthedocs.io/en/stable/examples/Using%20Interact.html)

Los datos se generan con fines educativos. Los controles ejecutan Python y necesitan el kernel activo.